# mast_ttest – Summary

This notebook applies **independent-sample t-tests** to evaluate whether **mast brand (Levitaz vs Chubanga)** influences boat performance, specifically **SOG (Speed Over Ground)**, under different run conditions.  
The analysis uses telemetry from `all_data.csv` filtered for **June 10, 2025 runs**.

---

## Inputs
- **Data**: `all_data.csv` containing time-series telemetry across runs.   
- **Helper functions**:  
  - `t_test(df1, df2, target="SOG")`: performs two-sample t-test, prints t-statistic, p-value, and interprets significance (`p < 0.05`).  
  - `print_run_stats(label, group1, group2, target)`: prints mean values of `target`, plus average and std of `SOG` for each group.

---

## Workflow

### Step 1: Load & filter data
- Restrict dataset to rows with timestamp starting `"2025-06-10"`.

### Step 2: Define groups of runs
- **Runs 1–5**: Karl on **Levitaz**, Gian on **Chubanga**.  
- **Runs 6–10**: Karl on **Chubanga**, Gian on **Levitaz**.  
- Subsets created: `data_10juin_first_runs`, `data_10juin_last_runs`.

### Step 3: Initial t-test (conditions check)
- Compare **TWS** between runs 1–5 and runs 6–10.  
- Report descriptive stats for both groups.

### Step 4: Karl Levitaz vs Karl Chubanga
- Subset Karl’s boat (or SenseBoard paired against Gian).  
- Run t-tests on **SOG**:
  - General (all legs).  
  - Upwind only (`TWA > 0`).  
  - Downwind only (`TWA ≤ 0`).  
- Use `print_run_stats` to show mast brand, mean SOG, std SOG per group.

### Step 5: Gian Chubanga vs Gian Levitaz
- Subset Gian’s boat (or SenseBoard paired against Karl).  
- Run t-tests on **SOG**:
  - General.  
  - Upwind only.  
  - Downwind only.  
- Use `print_run_stats` to show mast brand, mean SOG, std SOG per group.

---

## Output
- Printed results for each t-test: **t-statistic**, **p-value**, interpretation of significance.  
- Group summaries: mast brand used, mean SOG, std SOG.  
- Structured comparisons:  
  - **Karl**: Levitaz (runs 1–5) vs Chubanga (runs 6–10).  
  - **Gian**: Chubanga (runs 1–5) vs Levitaz (runs 6–10).  
  - Breakdown by **upwind** and **downwind**.

---

## Notes
- Significance threshold: `p < 0.05`.  
- Non-significant results suggest **data can be combined**; significant results indicate a **mast effect**.  

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import scipy.stats as stats

def t_test(df1, df2, target="SOG"):
    t_stat, p_value = stats.ttest_ind(df1[target].dropna(), df2[target].dropna())
    print(f"T-statistic: {t_stat:.3f}, p-value: {p_value:.15f}")
    
    # If p-value is less than 0.05, the difference is statistically significant
    if p_value < 0.05:
        print("The difference is statistically significant, keeping data split.")
    else:
        print("The difference is not statistically significant, keeping data combined.")

def print_run_stats(first_sentence, first_runs_df, last_runs_df, target):
    print("\n", first_sentence)

    if first_runs_df[target].dtype == "O":
        first_target = ", ".join(first_runs_df[target].dropna().unique())
        last_target = ", ".join(last_runs_df[target].dropna().unique())
    else:
        first_target = f"{first_runs_df[target].mean():.2f}"
        last_target = f"{last_runs_df[target].mean():.2f}"

    print(f"Mean {target } on the first group : {first_target}, "
          f"average SOG: {first_runs_df['SOG'].mean():.2f}, std SOG: {first_runs_df['SOG'].std():.2f}")
    
    print(f"Mean  {target } on the second group : {last_target}, "
          f"average SOG: {last_runs_df['SOG'].mean():.2f}, std SOG: {last_runs_df['SOG'].std():.2f}")



In [2]:
df = pd.read_csv("all_data.csv")

In [3]:
data_10juin = df[df["ISODateTimeUTC"].str.startswith("2025-06-10")]

## T test on the TWS between runs 1 to 5 where Karl is on the Levitaz and Gian is on the Chubanga and runs 6 to 10 is the other way around

In [4]:
first_runs = ["10_06_2025_Run1","10_06_2025_Run2","10_06_2025_Run3","10_06_2025_Run4","10_06_2025_Run5"]
data_10juin_first_runs = data_10juin[data_10juin["run"].isin(first_runs) ]

In [5]:
last_runs = ["10_06_2025_Run6","10_06_2025_Run7","10_06_2025_Run8","10_06_2025_Run9","10_06_2025_Run10"]
data_10juin_last_runs = data_10juin[data_10juin["run"].isin(last_runs) ]

In [6]:
t_test(data_10juin_first_runs,data_10juin_last_runs, target="TWS")
print_run_stats("Runs 1 to 5 VS Runs 6 to 10:", data_10juin_first_runs, data_10juin_last_runs, target="TWS")

T-statistic: -208.083, p-value: 0.000000000000000
The difference is statistically significant, keeping data split.

 Runs 1 to 5 VS Runs 6 to 10:
Mean TWS on the first group : 6.19, average SOG: 24.06, std SOG: 2.19
Mean  TWS on the second group : 8.06, average SOG: 24.57, std SOG: 2.16


## t test karl levi vs karl chub

In [7]:
only_karl_first_runs_levi = data_10juin_first_runs[
    (data_10juin_first_runs["boat_name"] == "Karl Maeder") |
    ((data_10juin_first_runs["boat_name"] == "SenseBoard") & 
     (data_10juin_first_runs["opponent_name"] == "Gian Stragiotti"))
]
only_karl_first_runs_levi.sample(5)

,ISODateTimeUTC,SecondsSince1970,Heel_Abs,Heel_Lwd,Lat,LatBow,LatCenter,LatStern,Leg,Line_C,...,interval_duration,mast_brand,gain_forward,gain_lateral,gain_vmg,Line_R2,Line_L2,Line_C2,side_line2,total_line2
59388,2025-06-10T12:29:01.151Z,1.749559e+09,54.8,54.8,43.533201,43.533199,43.533205,43.533211,NaN,102.200,...,73.306,Levi,-0.005692,-7.474989,-6.009457,6.400,7.000,102.200,13.400,115.600
64381,2025-06-10T12:44:50.150Z,1.749559e+09,58.3,58.3,43.531871,43.531870,43.531875,43.531881,NaN,96.300,...,69.201,Levi,-0.564895,-1.623020,-1.625698,2.284,4.900,96.300,7.184,103.484
61777,2025-06-10T12:36:58.656Z,1.749559e+09,45.5,45.5,43.533394,43.533392,43.533398,43.533404,NaN,110.200,...,62.101,Levi,6.063792,0.429774,4.278548,5.451,6.947,110.200,12.398,122.598
60702,2025-06-10T12:31:53.658Z,1.749559e+09,45.6,45.6,43.533333,43.533335,43.533329,43.533323,NaN,74.700,...,54.787,Levi,-12.581466,-7.514214,1.286000,1.615,3.200,74.700,4.815,79.515
65394,2025-06-10T12:47:37.752Z,1.749560e+09,47.9,47.9,43.532960,43.532962,43.532956,43.532949,NaN,79.007,...,48.401,Levi,-12.725321,0.877460,9.103607,3.291,3.724,79.007,7.015,86.022


In [8]:
only_karl_last_runs_chub = data_10juin_last_runs[
    (data_10juin_last_runs["boat_name"] == "Karl Maeder") |
    ((data_10juin_last_runs["boat_name"] == "SenseBoard") & 
     (data_10juin_last_runs["opponent_name"] == "Gian Stragiotti"))
]
only_karl_last_runs_chub.sample(5)

,ISODateTimeUTC,SecondsSince1970,Heel_Abs,Heel_Lwd,Lat,LatBow,LatCenter,LatStern,Leg,Line_C,...,interval_duration,mast_brand,gain_forward,gain_lateral,gain_vmg,Line_R2,Line_L2,Line_C2,side_line2,total_line2
77907,2025-06-10T13:42:35.858Z,1.749563e+09,54.6,54.6,43.530489,43.530490,43.530484,43.530478,NaN,93.6,...,46.883,Chub,-1.788173,2.773759,3.123639,3.6,5.297,93.6,8.897,102.497
74878,2025-06-10T13:33:43.460Z,1.749562e+09,63.2,63.2,43.530452,43.530451,43.530456,43.530462,NaN,86.3,...,62.598,Chub,14.144725,12.034995,18.481571,2.2,4.700,86.3,6.900,93.200
72280,2025-06-10T13:23:01.155Z,1.749562e+09,58.8,58.8,43.532689,43.532687,43.532693,43.532698,NaN,106.3,...,67.800,Chub,-3.465599,-2.154875,-3.871965,7.1,9.000,106.3,16.100,122.400
78210,2025-06-10T13:43:06.154Z,1.749563e+09,51.9,51.9,43.534177,43.534179,43.534172,43.534166,NaN,77.0,...,46.883,Chub,-13.913423,-2.266579,9.589413,3.7,4.700,77.0,8.400,85.400
75820,2025-06-10T13:36:04.657Z,1.749563e+09,40.0,40.0,43.533020,43.533022,43.533016,43.533010,NaN,62.4,...,47.895,Chub,-0.058575,-3.996144,-2.418149,2.1,4.000,62.4,6.100,68.500


In [9]:
t_test(only_karl_first_runs_levi,only_karl_last_runs_chub) #general
print("\nUpwind and downwind for Karl:")
print_run_stats("Karl on Levi VS Karl on Chub:", only_karl_first_runs_levi, only_karl_last_runs_chub, target="mast_brand")

T-statistic: -15.956, p-value: 0.000000000000000
The difference is statistically significant, keeping data split.

Upwind and downwind for Karl:

 Karl on Levi VS Karl on Chub:
Mean mast_brand on the first group : Levi, average SOG: 23.80, std SOG: 2.03
Mean  mast_brand on the second group : Chub, average SOG: 24.44, std SOG: 2.17


In [10]:
only_karl_first_runs_levi_upwind = only_karl_first_runs_levi[only_karl_first_runs_levi["TWA"]>0]
only_karl_last_runs_chub_upwind = only_karl_last_runs_chub[only_karl_last_runs_chub["TWA"]>0]
# upwind
print("\nUpwind for Karl:")
t_test(only_karl_first_runs_levi_upwind,only_karl_last_runs_chub_upwind)
print_run_stats("Karl on Levi upwind VS Karl on Chub upwind:", only_karl_first_runs_levi_upwind, only_karl_last_runs_chub_upwind, target="mast_brand")


Upwind for Karl:
T-statistic: -44.815, p-value: 0.000000000000000
The difference is statistically significant, keeping data split.

 Karl on Levi upwind VS Karl on Chub upwind:
Mean mast_brand on the first group : Levi, average SOG: 22.12, std SOG: 0.73
Mean  mast_brand on the second group : Chub, average SOG: 22.84, std SOG: 0.54


In [11]:
only_karl_first_runs_levi_downwind = only_karl_first_runs_levi[only_karl_first_runs_levi["TWA"] <= 0]
only_karl_last_runs_chub_downwind = only_karl_last_runs_chub[only_karl_last_runs_chub["TWA"] <= 0]
#downwind
print("\nDownwind for Karl:")
t_test(only_karl_first_runs_levi_downwind,only_karl_last_runs_chub_downwind)
print_run_stats("Karl on Levi downwind VS Karl on Chub downwind:", only_karl_first_runs_levi_downwind, only_karl_last_runs_chub_downwind, target="mast_brand")


Downwind for Karl:
T-statistic: -56.835, p-value: 0.000000000000000
The difference is statistically significant, keeping data split.

 Karl on Levi downwind VS Karl on Chub downwind:
Mean mast_brand on the first group : Levi, average SOG: 25.86, std SOG: 0.91
Mean  mast_brand on the second group : Chub, average SOG: 27.20, std SOG: 0.56


## t test Gian chub vs Gian levi

In [12]:
only_gian_first_runs_chub = data_10juin_first_runs[
    (data_10juin_first_runs["boat_name"] == "Gian Stragiotti") |
    ((data_10juin_first_runs["boat_name"] == "SenseBoard") & 
     (data_10juin_first_runs["opponent_name"] == "Karl Maeder"))
]
only_gian_first_runs_chub.sample(5)

,ISODateTimeUTC,SecondsSince1970,Heel_Abs,Heel_Lwd,Lat,LatBow,LatCenter,LatStern,Leg,Line_C,...,interval_duration,mast_brand,gain_forward,gain_lateral,gain_vmg,Line_R2,Line_L2,Line_C2,side_line2,total_line2
62618,2025-06-10T12:40:15.651Z,1.749559e+09,56.1,56.1,43.534746,43.534748,43.534741,43.534735,NaN,118.3,...,54.391,Chub,20.679054,-6.824304,-17.980094,6.5,8.1,118.3,14.6,132.9
60297,2025-06-10T12:32:07.954Z,1.749559e+09,53.8,53.8,43.535105,43.535107,43.535101,43.535095,1.0,102.0,...,54.787,Chub,-18.214376,-11.129538,1.309033,6.1,8.0,102.0,14.1,116.1
65598,2025-06-10T12:52:49.753Z,1.749560e+09,60.5,60.5,43.535859,43.535857,43.535863,43.535869,NaN,124.3,...,51.991,Chub,0.684093,-0.294246,0.195459,5.0,5.4,124.3,10.4,134.7
58799,2025-06-10T12:29:15.545Z,1.749559e+09,63.0,63.0,43.531811,43.531809,43.531815,43.531821,1.0,135.6,...,73.306,Chub,-3.869396,-7.935314,-8.678172,7.9,8.3,135.6,16.2,151.8
61132,2025-06-10T12:36:56.355Z,1.749559e+09,56.7,56.7,43.533518,43.533516,43.533522,43.533528,NaN,137.5,...,62.101,Chub,5.573557,-1.084440,2.772024,5.3,7.3,137.5,12.6,150.1


In [13]:
only_gian_last_runs_levi = data_10juin_last_runs[
    (data_10juin_last_runs["boat_name"] == "Gian Stragiotti") |
    ((data_10juin_last_runs["boat_name"] == "SenseBoard") & 
     (data_10juin_last_runs["opponent_name"] == "Karl Maeder"))
]
only_gian_last_runs_levi.sample(5)

,ISODateTimeUTC,SecondsSince1970,Heel_Abs,Heel_Lwd,Lat,LatBow,LatCenter,LatStern,Leg,Line_C,...,interval_duration,mast_brand,gain_forward,gain_lateral,gain_vmg,Line_R2,Line_L2,Line_C2,side_line2,total_line2
75238,2025-06-10T13:33:16.756Z,1.749562e+09,50.2,50.2,43.532941,43.532940,43.532946,43.532952,NaN,136.70,...,62.598,Levi,4.046563,7.780733,8.397328,8.100,10.900,136.70,19.000,155.700
77597,2025-06-10T13:40:23.956Z,1.749563e+09,56.7,56.7,43.532052,43.532050,43.532056,43.532062,NaN,112.60,...,66.197,Levi,-6.342450,5.970952,0.116748,8.049,8.242,112.60,16.291,128.891
77320,2025-06-10T13:39:56.256Z,1.749563e+09,67.0,67.0,43.534851,43.534850,43.534856,43.534861,NaN,135.50,...,66.197,Levi,-3.102382,3.763273,0.539029,12.400,13.300,135.50,25.700,161.200
76167,2025-06-10T13:35:51.453Z,1.749563e+09,53.1,53.1,43.531330,43.531331,43.531325,43.531319,NaN,117.35,...,47.895,Levi,1.826064,0.480467,-1.149359,10.100,11.100,117.35,21.200,138.550
76217,2025-06-10T13:35:56.453Z,1.749563e+09,65.0,65.0,43.531959,43.531961,43.531955,43.531948,NaN,99.10,...,47.895,Levi,0.763342,2.432538,0.898066,8.600,9.500,99.10,18.100,117.200


In [14]:
t_test(only_gian_first_runs_chub,only_gian_last_runs_levi) #GENERAL
print("\nUpwind and downwind for Gian:")
print_run_stats("Gian on chub VS Gian on levi:", only_gian_first_runs_chub, only_gian_last_runs_levi, target="mast_brand")

T-statistic: -8.918, p-value: 0.000000000000000
The difference is statistically significant, keeping data split.

Upwind and downwind for Gian:

 Gian on chub VS Gian on levi:
Mean mast_brand on the first group : Chub, average SOG: 24.32, std SOG: 2.30
Mean  mast_brand on the second group : Levi, average SOG: 24.70, std SOG: 2.14


In [15]:
only_gian_first_runs_chub_upwind = only_gian_first_runs_chub[only_gian_first_runs_chub["TWA"]>0]
only_gian_last_runs_levi_upwind = only_gian_last_runs_levi[only_gian_last_runs_levi["TWA"]>0]
print("\nUpwind for Gian:")
t_test(only_gian_first_runs_chub_upwind,only_gian_last_runs_levi_upwind) #upwind
print_run_stats("Gian on chub upwind VS Gian on levi upwind:", only_gian_first_runs_chub_upwind, only_gian_last_runs_levi_upwind, target="mast_brand")


Upwind for Gian:


T-statistic: -35.891, p-value: 0.000000000000000
The difference is statistically significant, keeping data split.

 Gian on chub upwind VS Gian on levi upwind:
Mean mast_brand on the first group : Chub, average SOG: 22.42, std SOG: 0.98
Mean  mast_brand on the second group : Levi, average SOG: 23.13, std SOG: 0.55


In [16]:
only_gian_first_runs_chub_downwind = only_gian_first_runs_chub[only_gian_first_runs_chub["TWA"] <= 0]
only_gian_last_runs_levi_downwind = only_gian_last_runs_levi[only_gian_last_runs_levi["TWA"] <= 0]
print("\nDownwind for Gian:")
t_test(only_gian_first_runs_chub_downwind,only_gian_last_runs_levi_downwind) #upwind
print_run_stats("Gian on chub downwind VS Gian on levi downwind:", only_gian_first_runs_chub_downwind, only_gian_last_runs_levi_downwind, target="mast_brand")


Downwind for Gian:
T-statistic: -32.900, p-value: 0.000000000000000
The difference is statistically significant, keeping data split.

 Gian on chub downwind VS Gian on levi downwind:
Mean mast_brand on the first group : Chub, average SOG: 26.65, std SOG: 0.86
Mean  mast_brand on the second group : Levi, average SOG: 27.41, std SOG: 0.58
